# Modeling — **SVM + TF-IDF (sklearn)** (Colab)

Driver lengkap pendamping IndoBERT: clone repo → `train_svm_full14k` (GridSearch 24
kombinasi, split kanonik) → tampilkan SELURUH proses (hasil grid, confusion matrix,
top fitur diskriminatif per kelas, contoh komentar + klasifikasi & analisis error).
Pilih dataset di **bagian 2** via `TAG`/`SUBSET` (default **balanced3k**). CPU cukup.

> **SVM Spark MLlib** TIDAK di sini — jalur Big Data, dijalankan di cluster (`src/spark/`).

## 1. Clone repo (privat) + dependensi

In [ ]:
import os
from getpass import getpass
GH_TOKEN = os.environ.get('GH_TOKEN') or getpass('GitHub PAT (classic, scope repo): ')
![ -d jokowi_sentiment_project ] || git clone https://{GH_TOKEN}@github.com/Arachnoida/jokowi_sentiment_project.git
%cd jokowi_sentiment_project
%pip install -q 'pymongo>=4' dnspython certifi scikit-learn matplotlib pandas numpy python-dotenv

## 2. Set MONGO_URI + jalankan training
`raw_comments` (label) + `processed_svm` (teks `svm` stemmed) dari Mongo Atlas
(IP allowlist `0.0.0.0/0`). Subset → split 2100/600/300.

In [ ]:
import os
from getpass import getpass
os.environ['MONGO_URI'] = os.environ.get('MONGO_URI') or getpass('MONGO_URI: ')

# === Pilih dataset ===
#   balanced 3k : TAG, SUBSET = 'balanced3k', 'outputs/labeling/balanced_3000.csv'
#   full 14k    : TAG, SUBSET = 'full14k', ''
TAG    = 'balanced3k'
SUBSET = 'outputs/labeling/balanced_3000.csv'

flags = f'--tag {TAG}' + (f' --subset {SUBSET}' if SUBSET else '')
!python -m src.modeling.train_svm_full14k $flags

## 3. Hasil grid search (24 kombinasi)
`ngram × min_df × C`, diurut peringkat macro-F1 di set validasi (`svm_<tag>_grid.csv`).

In [ ]:
import pandas as pd
g = pd.read_csv(f'outputs/reports/svm_{TAG}_grid.csv')
print(f'{len(g)} kombinasi dievaluasi. 8 teratas:')
display(g.head(8))

## 4. Confusion matrix + ringkasan metrik test

In [ ]:
import json
from IPython.display import Image, display
m = json.load(open(f'outputs/reports/svm_{TAG}_metrics.json'))['test']
print(f"Akurasi : {m['accuracy']}  | macro-F1: {m['macro_f1']}")
for l in ['Negatif','Netral','Positif']:
    print(f"  {l:<8} F1={m['per_class'][l]['f1']:.3f} (n={m['per_class'][l]['support']})")
print('best params:', m.get('best_params'))
display(Image(f'outputs/reports/svm_{TAG}_confusion.png'))

## 5. Top fitur diskriminatif per kelas
Kata/bigram dengan koefisien LinearSVC tertinggi tiap kelas (`svm_<tag>_top_features.csv`)
— interpretabilitas: apa yang 'menarik' prediksi ke tiap sentimen.

In [ ]:
tf = pd.read_csv(f'outputs/reports/svm_{TAG}_top_features.csv')
tab = {}
for lab in ['Negatif','Netral','Positif']:
    tab[lab] = tf[tf.kelas==lab].sort_values('rank')['fitur'].head(10).tolist()
display(pd.DataFrame(tab))

## 6. Contoh komentar + hasil klasifikasi (+ analisis error)
Dari test set (`svm_<tag>_predictions.csv`). ❌ = salah klasifikasi.

In [ ]:
p = pd.read_csv(f'outputs/reports/svm_{TAG}_predictions.csv')
salah = (~p['benar']).sum()
print(f'Test: {len(p)} komentar | benar {p["benar"].sum()} | salah {salah} | akurasi {p["benar"].mean():.4f}')

def tampil(df, n):
    for _, r in df.head(n).iterrows():
        mark = '✅' if r['benar'] else '❌'
        t = (str(r['text'])[:110]).replace(chr(10), ' ')
        print(f"{mark} asli={r['label_asli']:<7} pred={r['prediksi']:<7} | {t}")

print(chr(10)+'— Contoh SALAH (analisis error) —')
tampil(p[~p['benar']], 12)
print(chr(10)+'— Contoh BENAR —')
tampil(p[p['benar']], 8)

p['teks'] = p['text'].astype(str).str.slice(0, 90)
print(chr(10)+'Tabel (20 pertama):')
display(p[['label_asli','prediksi','benar','teks']].head(20))

## 7. (Opsional) Bandingkan 3 model

In [ ]:
suffix = '' if TAG == 'full14k' else f'_{TAG}'
need = [f'outputs/reports/svm_{TAG}_metrics.json',
        f'outputs/reports/svm_spark{suffix}_metrics.json',
        f'outputs/reports/indobert{suffix}_metrics.json']
if all(os.path.exists(x) for x in need):
    !python -m src.modeling.compare_models --tag $TAG
    display(Image(f'outputs/reports/model_comparison{"" if TAG=="full14k" else "_"+TAG}_accuracy.png'))
else:
    print('Lewati: butuh ketiga JSON. Ada:', [x for x in need if os.path.exists(x)])

In [ ]:
# Unduh artefak SVM (untuk di-commit ke outputs/reports/ lokal)
from google.colab import files
for f in [f'svm_{TAG}_metrics.json', f'svm_{TAG}_confusion.png',
          f'svm_{TAG}_grid.csv', f'svm_{TAG}_top_features.csv', f'svm_{TAG}_predictions.csv']:
    files.download(f'outputs/reports/{f}')